# Consolidado de Indicadores SISAV2

Produce una tabla unica con las columnas-indicador necesarias para los graficos del dashboard.
El equipo de visualizacion consume directamente el archivo exportado.

**Enfoque selectivo**: solo extraemos las columnas que alimentan indicadores.
No procesamos las planillas completas.

---

In [ ]:
import sys
import re
from pathlib import Path
import warnings

_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / ".."]
PROJECT_ROOT = next((p.resolve() for p in _candidates if (p / "src" / "__init__.py").exists()), Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
sns.set_theme(style="whitegrid", font_scale=0.9)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 80)

DATA_DIR = Path("/Users/ignacioramirez/Desktop/Proyecto VCM/drive-download-20260623T174348Z-3-001/iniciativas_a_excel/")
OUTPUT_DIR = PROJECT_ROOT / "data" / "clean"

---
## 1. Esquema canonico de indicadores

Estas son las unicas columnas que extraemos. Todo lo demas se ignora.

In [ ]:
ESQUEMA_CANONICO = [
    "codigo",
    "facultad",
    "carrera",
    "nombre_iniciativa",
    "estado_sisav",
    "semestre_ejecucion",
    "dominios_disciplinares",
    "area_generica",
    "competencia_sello",
    "actividad",
    "ciclo_modelo_educativo",
    "cantidad_act_planificadas",
    "cantidad_act_ejecutadas",
    "n_participantes",
    "ods",
]

COLS_LINAJE = ["_archivo_origen", "_instrumento", "_anio"]

print(f"Columnas-indicador: {len(ESQUEMA_CANONICO)}")
print(f"Columnas de linaje: {len(COLS_LINAJE)}")
for c in ESQUEMA_CANONICO:
    print(f"  - {c}")

---
## 2. Mapeo de columnas por familia de formato

Cada familia tiene nombres de columna distintos. El diccionario mapea
`columna_origen -> columna_canonica`. Solo se incluyen las que existen
en la fuente; las ausentes se rellenan con NaN automaticamente.

### Familia A: Legacy 2022-2023 (EXT, FCR, VEDP)
Hojas: PLANILLA. Columnas con nomenclatura antigua.

In [ ]:
# Familia A: EXT/FCR/VEDP 2022-2023
# Hoja: PLANILLA, nombres cortos, tiene Act Ejecutadas y Total Asistentes
# R-010: "Area" NO son dominios disciplinares, son categorias genericas -> area_generica
MAPEO_FAMILIA_A = {
    "Codigo ": "codigo",
    "Facultad": "facultad",
    "Carrera": "carrera",
    "Nombre": "nombre_iniciativa",
    "Estado SISAV": "estado_sisav",
    "Semestre Ejecución": "semestre_ejecucion",
    "Area": "area_generica",
    "Sello Institucional": "competencia_sello",
    "Actividad": "actividad",
    "Ciclo Modelo Educativo": "ciclo_modelo_educativo",
    "Cantidad Act Planificadas": "cantidad_act_planificadas",
    "Cantidad de Act Ejecutadas": "cantidad_act_ejecutadas",
    "Cantidad de Act Eje": "cantidad_act_ejecutadas",  # variante FCR
    "Total Asistentes": "n_participantes",
    "Identifica ODS": "ods",
}

ARCHIVOS_FAMILIA_A = [
    "Plantilla EXT 2022.xlsm",
    "Plantilla EXT 2023.xlsm",
    "Plantilla FCR 2022.xlsm",
    "Plantilla FCR 2023.xlsm",
    "Plantilla VEDP 2022.xlsm",
    "Plantilla VEDP 2023.xlsm",
]

print(f"Familia A: {len(ARCHIVOS_FAMILIA_A)} archivos, {len(MAPEO_FAMILIA_A)} columnas mapeadas")
print("Mapeo:")
for orig, canon in MAPEO_FAMILIA_A.items():
    print(f"  {orig:<35s} -> {canon}")

### Familia B: 2024+ Planilla (EXT, VEDP, UTG, Centralizadas)
Hojas: Planilla. Nomenclatura alineada con exports SISAV2.

In [ ]:
# Familia B: 2024 y Centralizadas
# Hoja: Planilla, nombres alineados con export plano, sin Act Ejecutadas
MAPEO_FAMILIA_B = {
    "Codigo ": "codigo",
    "Facultad": "facultad",
    "Carrera": "carrera",
    "Nombre de la iniciativa": "nombre_iniciativa",
    "Nombre": "nombre_iniciativa",  # variante UTG
    "Estado SISAV": "estado_sisav",
    "Semestre Ejecución": "semestre_ejecucion",
    "Dominios disciplinares": "dominios_disciplinares",
    "Competencia Sello": "competencia_sello",
    "Actividad": "actividad",
    "Ciclo del modelo educativo": "ciclo_modelo_educativo",
    "Cantidad Act Planificadas": "cantidad_act_planificadas",
    "N estudiantes": "n_participantes",
    "Total": "n_participantes",  # fallback si N estudiantes no existe
    "ODS": "ods",
}

ARCHIVOS_FAMILIA_B = [
    "Plantilla EXT 2024.xlsm",
    "Plantilla UTG2024.xlsm",
    "Plantilla VEDP 2024.xlsm",
    "Plantilla Iniciativas Centralizadas.xlsm",
    "Plantilla Iniciativas Centralizadas 2025.xlsm",
]

print(f"Familia B: {len(ARCHIVOS_FAMILIA_B)} archivos, {len(MAPEO_FAMILIA_B)} columnas mapeadas")
print("Mapeo:")
for orig, canon in MAPEO_FAMILIA_B.items():
    print(f"  {orig:<35s} -> {canon}")

### Familia C: Exports expandidos 2025 (.xlsx, hoja Postulaciones)
Formato nuevo con participantes desagregados y campos curriculares.

In [ ]:
# Familia C: Exports expandidos 2025
# Hoja: Postulaciones, formato nuevo con participantes desagregados
MAPEO_FAMILIA_C = {
    "Código": "codigo",
    "CÓDIGO SISAV": "codigo",
    "Código SISAV": "codigo",
    "Facultad": "facultad",
    "Carrera": "carrera",
    "Nombre de iniciativa": "nombre_iniciativa",
    "Estado SISAV": "estado_sisav",
    "Semestre de ejecución de la iniciativa": "semestre_ejecucion",
    "Dominios disciplinares": "dominios_disciplinares",
    "Competencia sello": "competencia_sello",
    "Actividad": "actividad",
    "Ciclo del modelo educativo": "ciclo_modelo_educativo",
    "Ciclo": "ciclo_modelo_educativo",  # variante EA
    "Cantidad de actividades planificadas": "cantidad_act_planificadas",
    "N total de participantes de la iniciativa": "n_participantes",
    "ODS": "ods",
}

ARCHIVOS_FAMILIA_C = [
    ("PRE_GRADO__conv32__Convocatoria_proyecto_VcM_Extensión_Académica_2025.xlsx", "Postulaciones"),
    ("PRE_GRADO__conv39__Convocatoria_proyecto_Vinculación_con_el_Entorno_Disciplinar_Profesional_2025.xlsx", "Postulaciones"),
    ("PRE_GRADO__conv38__Convocatoria_proyecto_VcM_Vinculación_con_Titulados_2025.xlsx", "Postulaciones"),
]

print(f"Familia C: {len(ARCHIVOS_FAMILIA_C)} archivos, {len(MAPEO_FAMILIA_C)} columnas mapeadas")
print("Mapeo:")
for orig, canon in MAPEO_FAMILIA_C.items():
    print(f"  {orig:<50s} -> {canon}")

---
## 3. Funcion de extraccion

Dado un archivo y su mapeo, lee la hoja principal, extrae solo las columnas-indicador
renombradas al canonico, y deriva instrumento/anio del nombre del archivo.

In [ ]:
def detectar_instrumento(filename: str) -> str:
    """Detecta el instrumento desde el nombre del archivo."""
    f = filename.upper()
    if "VEDP" in f or "VINCULACIÓN_CON_EL_ENTORNO" in f or "VINCULACION_CON_EL_ENTORNO" in f:
        return "VEDP"
    if "VCT" in f or "VINCULACIÓN_CON_TITULADOS" in f or "VINCULACION_CON_TITULADOS" in f or "TITULADOS" in f:
        return "VT"
    if "EXT" in f or "EXTENSIÓN" in f or "EXTENSION" in f:
        return "EXTENSION"
    if "FCR" in f:
        return "FCR"
    if "UTG" in f:
        return "UTG"
    if "CENTRALIZAD" in f:
        return "CENTRALIZADO"
    return "OTRO"


def detectar_anio(filename: str) -> int | None:
    """Extrae el anio de 4 digitos del nombre del archivo."""
    match = re.search(r"(20[12]\d)", filename)
    return int(match.group(1)) if match else None


def extraer_indicadores(filepath: Path, mapeo: dict, hoja: str = None) -> pd.DataFrame:
    """Lee un archivo y extrae las columnas-indicador mapeadas al canonico."""
    # Determinar hoja
    if hoja is None:
        xl = pd.ExcelFile(filepath, engine="openpyxl")
        hoja = next((s for s in xl.sheet_names if s.upper().strip() == "PLANILLA"), xl.sheet_names[0])

    df = pd.read_excel(filepath, sheet_name=hoja, engine="openpyxl")

    # Aplicar mapeo: buscar columnas presentes, resolviendo conflictos por prioridad
    resultado = {}
    for col_origen, col_canon in mapeo.items():
        if col_origen in df.columns and col_canon not in resultado:
            resultado[col_canon] = df[col_origen]

    df_out = pd.DataFrame(resultado)

    # Rellenar columnas faltantes con NaN
    for col in ESQUEMA_CANONICO:
        if col not in df_out.columns:
            df_out[col] = pd.NA

    # Ordenar columnas al canonico
    df_out = df_out[ESQUEMA_CANONICO]

    # Linaje
    df_out["_archivo_origen"] = filepath.name
    df_out["_instrumento"] = detectar_instrumento(filepath.name)
    df_out["_anio"] = detectar_anio(filepath.name)

    # Filtrar filas completamente vacias (padding del Excel)
    cols_dato = [c for c in ESQUEMA_CANONICO if c != "codigo"]
    df_out = df_out.dropna(subset=["codigo"], how="all")

    return df_out


print("Funcion extraer_indicadores definida.")
# Test rapido
test = extraer_indicadores(DATA_DIR / ARCHIVOS_FAMILIA_A[0], MAPEO_FAMILIA_A)
print(f"Test Familia A ({ARCHIVOS_FAMILIA_A[0]}): {test.shape[0]} filas, {test.shape[1]} cols")
print(test.head(3).to_string(index=False))

---
## 4. Consolidacion

Procesamos todos los archivos de las tres familias y apilamos en un solo DataFrame.

In [ ]:
frames = []
errores = []

# Familia A
for fname in ARCHIVOS_FAMILIA_A:
    try:
        df_f = extraer_indicadores(DATA_DIR / fname, MAPEO_FAMILIA_A)
        frames.append(df_f)
        print(f"  OK {fname}: {len(df_f)} filas")
    except Exception as e:
        errores.append((fname, str(e)))
        print(f"  ERROR {fname}: {e}")

# Familia B
for fname in ARCHIVOS_FAMILIA_B:
    try:
        df_f = extraer_indicadores(DATA_DIR / fname, MAPEO_FAMILIA_B)
        frames.append(df_f)
        print(f"  OK {fname}: {len(df_f)} filas")
    except Exception as e:
        errores.append((fname, str(e)))
        print(f"  ERROR {fname}: {e}")

# Familia C
for fname, hoja in ARCHIVOS_FAMILIA_C:
    try:
        df_f = extraer_indicadores(DATA_DIR / fname, MAPEO_FAMILIA_C, hoja=hoja)
        frames.append(df_f)
        print(f"  OK {fname[:50]}: {len(df_f)} filas")
    except Exception as e:
        errores.append((fname, str(e)))
        print(f"  ERROR {fname}: {e}")

print(f"\nTotal frames: {len(frames)}, errores: {len(errores)}")

df_consolidado = pd.concat(frames, ignore_index=True)
print(f"Consolidado: {df_consolidado.shape[0]} filas x {df_consolidado.shape[1]} columnas")

---
## 5. Limpieza minima

Reutilizamos las reglas del pipeline donde aplican (espacios, puntos finales).
No aplicamos reglas de fecha porque no tenemos esa columna en el consolidado de indicadores.

In [ ]:
n_antes = len(df_consolidado)
print(f"Shape pre-limpieza: {df_consolidado.shape}")
print()

# --- Correccion 1: Verificar separacion area/dominios ---
print("=== C1: Separacion area_generica / dominios_disciplinares ===")
for anio in sorted(df_consolidado["_anio"].dropna().unique().astype(int)):
    n_dom = df_consolidado[df_consolidado["_anio"] == anio]["dominios_disciplinares"].notna().sum()
    n_area = df_consolidado[df_consolidado["_anio"] == anio]["area_generica"].notna().sum()
    print(f"  {anio}: dominios={n_dom}, area_generica={n_area}")
print()

# --- Correccion 2: R-011 Limpiar artefacto vscode-resource en dominios 2024-2025 ---
print("=== C2: Limpiar artefacto vscode-resource (R-011) ===")
ARTEFACTO = r"\[/\]\(https://file\+\.vscode-resource\.vscode-cdn\.net/\)"
col_dom = "dominios_disciplinares"
mask_dom = df_consolidado[col_dom].notna() & df_consolidado[col_dom].apply(lambda x: isinstance(x, str))
n_artefacto = df_consolidado.loc[mask_dom, col_dom].str.contains(ARTEFACTO, regex=True, na=False).sum()
print(f"  Valores con artefacto antes: {n_artefacto}")

if n_artefacto > 0:
    ejemplo_antes = df_consolidado.loc[mask_dom, col_dom][
        df_consolidado.loc[mask_dom, col_dom].str.contains(ARTEFACTO, regex=True, na=False)
    ].iloc[0][:120]
    print(f"  Ejemplo antes: {ejemplo_antes}...")

    df_consolidado.loc[mask_dom, col_dom] = df_consolidado.loc[mask_dom, col_dom].apply(
        lambda x: re.sub(ARTEFACTO, "; ", x) if isinstance(x, str) else x
    )

    ejemplo_despues = df_consolidado.loc[mask_dom, col_dom][mask_dom].dropna().iloc[0][:120]
    n_despues = df_consolidado.loc[mask_dom, col_dom].str.contains(ARTEFACTO, regex=True, na=False).sum()
    print(f"  Valores con artefacto despues: {n_despues}")
    print(f"  Ejemplo despues: {ejemplo_despues}...")
print()

# --- Correccion 3: R-012 Normalizar separadores multivalor a "; " ---
print("=== C3: Normalizar separadores multivalor (R-012) ===")
COLS_MULTIVALOR = ["dominios_disciplinares", "area_generica", "competencia_sello", "ods"]

def normalizar_separador(val: str) -> str:
    """Unifica separadores (coma, punto y coma, variantes) a '; ' y limpia items."""
    val = re.sub(r"\s*;\s*", "; ", val)
    val = re.sub(r"\s*,\s*", "; ", val)
    items = [item.strip() for item in val.split("; ")]
    items = [item for item in items if item]
    return "; ".join(items)

for col in COLS_MULTIVALOR:
    mask = df_consolidado[col].notna() & df_consolidado[col].apply(lambda x: isinstance(x, str))
    n_con_coma = df_consolidado.loc[mask, col].str.contains(",", na=False).sum()
    df_consolidado.loc[mask, col] = df_consolidado.loc[mask, col].apply(normalizar_separador)
    print(f"  {col}: {n_con_coma} valores tenian comas como separador (normalizados)")

# R-013: Normalizar separador " / " (espacio-barra-espacio) a "; "
print("\n=== C3b: Normalizar separador ' / ' a '; ' (R-013) ===")
for col in COLS_MULTIVALOR:
    mask = df_consolidado[col].notna() & df_consolidado[col].apply(lambda x: isinstance(x, str))
    n_con_slash = df_consolidado.loc[mask, col].str.contains(r" / ", regex=False, na=False).sum()
    df_consolidado.loc[mask, col] = df_consolidado.loc[mask, col].apply(
        lambda x: x.replace(" / ", "; ") if isinstance(x, str) else x
    )
    # Re-limpiar: strip items y eliminar vacios
    df_consolidado.loc[mask, col] = df_consolidado.loc[mask, col].apply(
        lambda x: "; ".join(item.strip() for item in x.split("; ") if item.strip()) if isinstance(x, str) else x
    )
    print(f"  {col}: {n_con_slash} valores tenian ' / ' como separador (normalizados)")

# Eliminar puntos finales de cada item
for col in COLS_MULTIVALOR:
    mask = df_consolidado[col].notna() & df_consolidado[col].apply(lambda x: isinstance(x, str))
    df_consolidado.loc[mask, col] = df_consolidado.loc[mask, col].apply(
        lambda x: "; ".join(item.rstrip(".") for item in x.split("; "))
    )

# Verificacion post-R-013
print("\n=== Verificacion post-R-013 ===")
for col in COLS_MULTIVALOR:
    mask = df_consolidado[col].notna() & df_consolidado[col].apply(lambda x: isinstance(x, str))
    n_slash_espacios = df_consolidado.loc[mask, col].str.contains(r" / ", regex=False, na=False).sum()
    n_slash_sin_esp = df_consolidado.loc[mask, col].str.contains(r"[^ ]/[^ ]", regex=True, na=False).sum()
    print(f"  {col}: ' / ' residual={n_slash_espacios}, '/' sin espacios (legitimo)={n_slash_sin_esp}")

print("\nMuestra dominios_disciplinares (2025):")
muestra = df_consolidado[
    (df_consolidado["_anio"] == 2025) & df_consolidado["dominios_disciplinares"].notna()
]["dominios_disciplinares"].head(5)
for v in muestra:
    print(f"  {str(v)[:120]}")

print(f"\nFilas tras R-013: {len(df_consolidado)} (debe ser 593)")
print()

# --- Correccion 4: Normalizar texto general ---
print("=== C4: Normalizar texto general ===")
cols_texto = [c for c in ESQUEMA_CANONICO if c not in ("cantidad_act_planificadas", "cantidad_act_ejecutadas", "n_participantes")]
n_modificadas = 0
for col in cols_texto:
    mask = df_consolidado[col].notna() & df_consolidado[col].apply(lambda x: isinstance(x, str))
    antes = df_consolidado.loc[mask, col].copy()
    # Strip, colapso de espacios, eliminacion de tabs/newlines
    df_consolidado.loc[mask, col] = df_consolidado.loc[mask, col].apply(
        lambda x: " ".join(x.split())
    )
    n_modificadas += (antes != df_consolidado.loc[mask, col]).sum()
print(f"  Valores con espacios/tabs/newlines corregidos: {n_modificadas}")
print()

# --- Correccion 5: Estandarizar tipos ---
print("=== C5: Estandarizar tipos ===")
for col in ["cantidad_act_planificadas", "cantidad_act_ejecutadas", "n_participantes"]:
    df_consolidado[col] = pd.to_numeric(df_consolidado[col], errors="coerce")

df_consolidado["codigo"] = (
    df_consolidado["codigo"].astype(str)
    .replace({"nan": None, "<NA>": None, "None": None, "nan": None})
)
df_consolidado["_anio"] = pd.to_numeric(df_consolidado["_anio"], errors="coerce").astype("Int64")

print(df_consolidado.dtypes.to_string())
print()

# --- Correccion 6: Quitar filas basura ---
print("=== C6: Quitar filas basura ===")
mask_basura = df_consolidado["codigo"].isna() | df_consolidado["nombre_iniciativa"].isna()
n_basura = mask_basura.sum()
df_consolidado = df_consolidado[~mask_basura].reset_index(drop=True)
print(f"  Filas eliminadas (sin codigo o nombre): {n_basura}")
print(f"  Filas finales: {len(df_consolidado)}")
print()

print(f"Shape post-limpieza: {df_consolidado.shape}")
print(f"Filas eliminadas total: {n_antes - len(df_consolidado)}")

---
## 6. Reporte de cobertura

Por columna-indicador: % poblado global y desde que anio esta disponible.

In [ ]:
cobertura = []
for col in ESQUEMA_CANONICO:
    n_total = len(df_consolidado)
    n_poblado = df_consolidado[col].notna().sum()
    pct = n_poblado / n_total * 100

    # Desde que anio tiene datos
    con_dato = df_consolidado[df_consolidado[col].notna()]
    anio_min = con_dato["_anio"].min() if len(con_dato) > 0 else None
    anio_max = con_dato["_anio"].max() if len(con_dato) > 0 else None

    cobertura.append({
        "columna": col,
        "n_poblado": n_poblado,
        "pct_poblado": round(pct, 1),
        "desde_anio": int(anio_min) if pd.notna(anio_min) else "-",
        "hasta_anio": int(anio_max) if pd.notna(anio_max) else "-",
    })

df_cob = pd.DataFrame(cobertura)
print("Cobertura del consolidado:")
print(df_cob.to_string(index=False))
print(f"\nTotal filas: {len(df_consolidado)}")
print(f"Anios cubiertos: {sorted(df_consolidado['_anio'].dropna().unique().astype(int))}")

In [ ]:
# Cobertura por anio (heatmap)
cob_anio = df_consolidado.groupby("_anio")[ESQUEMA_CANONICO].apply(lambda g: g.notna().mean() * 100)

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(cob_anio.T, annot=True, fmt=".0f", cmap="YlGn", ax=ax, linewidths=0.5, vmin=0, vmax=100)
ax.set_title("% de cobertura por columna-indicador y año", fontsize=11)
ax.set_xlabel("Anio")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

---
## 7. Graficos de prueba

Demuestran que el consolidado es consumible. No son los graficos finales del dashboard.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 7.1 Iniciativas por anio
ax = axes[0, 0]
por_anio = df_consolidado.groupby("_anio").size().reset_index(name="n")
ax.bar(por_anio["_anio"].astype(int), por_anio["n"], color="#42a5f5")
ax.set_xlabel("Anio")
ax.set_ylabel("N de iniciativas")
ax.set_title("Iniciativas por anio")

# 7.2 Iniciativas por instrumento
ax = axes[0, 1]
por_instr = df_consolidado["_instrumento"].value_counts()
ax.barh(por_instr.index, por_instr.values, color="#66bb6a")
ax.set_xlabel("N de iniciativas")
ax.set_title("Iniciativas por instrumento")

# 7.3 Iniciativas por facultad (top 8)
ax = axes[1, 0]
por_fac = df_consolidado["facultad"].value_counts().head(8)
ax.barh(por_fac.index, por_fac.values, color="#ffa726")
ax.set_xlabel("N de iniciativas")
ax.set_title("Iniciativas por facultad (top 8)")

# 7.4 Participantes por anio
ax = axes[1, 1]
part_anio = df_consolidado.groupby("_anio")["n_participantes"].sum().reset_index()
part_anio = part_anio[part_anio["n_participantes"] > 0]
ax.bar(part_anio["_anio"].astype(int), part_anio["n_participantes"], color="#ab47bc")
ax.set_xlabel("Anio")
ax.set_ylabel("Total participantes")
ax.set_title("Participantes por anio")

plt.suptitle("Graficos de prueba - Consolidado de Indicadores", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

---
## 8. Seccion I19: actividades ejecutadas (solo 2022-2023)

El dato de actividades ejecutadas solo existe en la Familia A (2022-2023).
El indicador de cumplimiento (ejecutadas / planificadas) solo se calcula para ese periodo.
Esto no bloquea el consolidado; se documenta como limitacion conocida.

In [ ]:
df_i19 = df_consolidado[
    df_consolidado["cantidad_act_ejecutadas"].notna() &
    df_consolidado["cantidad_act_planificadas"].notna() &
    (df_consolidado["cantidad_act_planificadas"] > 0)
].copy()

df_i19["pct_cumplimiento"] = (df_i19["cantidad_act_ejecutadas"] / df_i19["cantidad_act_planificadas"] * 100).round(1)

print(f"Iniciativas con dato de cumplimiento (I19): {len(df_i19)}")
print(f"Anios disponibles: {sorted(df_i19['_anio'].unique().astype(int))}")
print(f"Instrumentos: {df_i19['_instrumento'].unique().tolist()}")
print()
print(f"Cumplimiento promedio: {df_i19['pct_cumplimiento'].mean():.1f}%")
print(f"Cumplimiento mediano: {df_i19['pct_cumplimiento'].median():.1f}%")
print()

# Distribucion por anio
resumen_i19 = df_i19.groupby("_anio").agg(
    n_iniciativas=("codigo", "count"),
    cumplimiento_promedio=("pct_cumplimiento", "mean"),
    cumplimiento_mediano=("pct_cumplimiento", "median"),
).round(1)
print("Cumplimiento por anio:")
print(resumen_i19.to_string())

---
## 9. Exportar consolidado

Se exporta a `data/clean/consolidado_indicadores.csv` y `.parquet`
para consumo directo del equipo de visualizacion.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

out_csv = OUTPUT_DIR / "consolidado_indicadores.csv"
out_parquet = OUTPUT_DIR / "consolidado_indicadores.parquet"

df_consolidado.to_csv(out_csv, index=False)
df_consolidado.to_parquet(out_parquet, index=False, engine="pyarrow")

print(f"Exportado: {out_csv} ({len(df_consolidado)} filas)")
print(f"Exportado: {out_parquet}")
print()
print("Columnas del archivo exportado:")
for c in df_consolidado.columns:
    print(f"  - {c}")